##### Instructions

Complete every code location marked **`TODO`**. Run the notebook from top to bottom after completing each section.

By the end of this exercise, you should be able to:

1. implement centroid initialization, assignment, and update steps in K-means;
2. calculate distortion and the silhouette score;
3. compare a from-scratch implementation with scikit-learn;
4. implement the E-step and M-step of a Gaussian Mixture Model (GMM);
5. monitor log-likelihood and use it as a convergence criterion.

**Rules:** Do not call `sklearn.cluster.KMeans` inside the from-scratch K-means class, and do not call `sklearn.mixture.GaussianMixture` inside the from-scratch GMM class. Small numerical constants such as `1e-9` may be added when needed for stability.

## a) Clustering using K-means

### Program 1 - Implementing K-means Clustering from scratch and using scikit learn

#### AIM

To implement K-means clustering from scratch, evaluate the resulting clusters, and compare the implementation with scikit-learn.

#### K-means algorithm

1. Select $k$ observations as the initial centroids.
2. Assign every observation to its nearest centroid.
3. Replace each centroid by the mean of the observations assigned to it.
4. Repeat steps 2–3 until the centroids stop changing (or a maximum number of iterations is reached).

#### Distortion

In this exercise, distortion is the mean Euclidean distance between every observation and its assigned centroid:

$$D = \frac{1}{N}\sum_{i=1}^{N}\lVert\mathbf{x}_i-\boldsymbol{\mu}_{z_i}\rVert_2.$$

#### Silhouette score

For observation $i$, let $a(i)$ be its mean distance to the other points in its own cluster and let $b(i)$ be the smallest mean distance from it to any other cluster. Then

$$s(i)=\frac{b(i)-a(i)}{\max\{a(i),b(i)\}}, \qquad S=\frac{1}{N}\sum_i s(i).$$

A score near $1$ indicates well-separated clusters, while a score near $-1$ indicates that many observations may be assigned to the wrong cluster.

#### Part 1 - Defining class for KMeans

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

np.set_printoptions(precision=4, suppress=True)

In [ ]:
class KMeans:
    def __init__(self, n_clusters, max_iter=300, tol=1e-4):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol

    def fit(self, train_x, random_state=0):
        m = train_x.shape[0]
        rng = np.random.default_rng(random_state)

        # TODO 1: Select `n_clusters` distinct observations as initial centroids.
        # Hint: rng.choice(..., replace=False)
        self.centroids = None

        for iteration in range(self.max_iter):
            old_centroids = self.centroids.copy()

            # TODO 2: Assign each training observation to its nearest centroid.
            self.labels = None

            # TODO 3: Update each centroid using the mean of its assigned points.
            # Keep the old centroid if a cluster receives no observations.
            for cluster_id in range(self.n_clusters):
                pass

            # TODO 4: Stop when the centroid displacement is at most `self.tol`.
            # Hint: np.linalg.norm(self.centroids - old_centroids)
            if False:
                break

        self.n_iter_ = iteration + 1
        return self

    def predict(self, points):
        # TODO 5: Calculate point-to-centroid distances and return the index
        # of the nearest centroid for every point.
        pass

#### Part 2 — Defining evaluation metrics

Complete both functions without calling their scikit-learn equivalents. Test the edge cases: a single cluster, a singleton cluster, and more than two clusters.

In [ ]:
def distortion(X, labels, centroids):
    # TODO 6: Select the assigned centroid for every observation and return
    # the mean Euclidean distance to that centroid.
    pass


def silhouette_score(X, labels):
    # TODO 7: Implement the mean silhouette score from its definition.
    # Suggested steps:
    #   a. Build the pairwise distance matrix with cdist(X, X).
    #   b. For each point, calculate a(i) from its own cluster.
    #   c. Calculate the mean distance to every other cluster and take b(i).
    #   d. Calculate s(i), treating a singleton cluster's score as 0.
    pass

#### Part 3 — Loading the Iris dataset

We use the copy bundled with scikit-learn so that this notebook does not depend on a separate CSV file. The species column is retained only for the later visual comparison; it is not supplied to either clustering algorithm.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
iris_df = iris.frame.rename(columns={
    "sepal length (cm)": "s_len",
    "sepal width (cm)": "s_wid",
    "petal length (cm)": "p_len",
    "petal width (cm)": "p_wid",
    "target": "species_code",
})
iris_df["species"] = iris_df["species_code"].map(dict(enumerate(iris.target_names)))
iris_df = iris_df.drop(columns="species_code")
names = ["s_len", "s_wid", "p_len", "p_wid", "species"]

display(iris_df.head())
print(
    f"The dataset contains {iris_df.shape[0]} records and "
    f"{iris_df.shape[1] - 1} numerical features.",
    iris_df["species"].value_counts(),
    sep="\n",
)

#### Part 4 - Implementing K-means Clustering

In [ ]:
train_x = iris_df.drop(columns="species").to_numpy()

# TODO 8: Set k to the number of known Iris species, fit your KMeans model,
# and construct `all_df` for the plotting cell below.
k = None
kmeans = None

# The completed objects should satisfy the following checks.
assert kmeans is not None, "TODO 8: create and fit the KMeans model"
assert kmeans.centroids.shape == (k, train_x.shape[1])
assert kmeans.labels.shape == (train_x.shape[0],)

all_df = iris_df.copy(deep=True)
centroids = pd.DataFrame(kmeans.centroids, columns=names[:-1])
centroids["cluster"] = "centroid"
all_df["cluster"] = kmeans.labels.astype(str)
all_df = pd.concat([all_df, centroids], ignore_index=True)

#### Part 5 - Comparing the Clustering results with original classes

In [ ]:
plt.figure(figsize=(15,6))
plt.suptitle("classes vs clusters")
plt.subplot(121)
plt.title("Original Classes")
sns.scatterplot(data=all_df,x="s_len",y="s_wid",hue="species",s=80)
plt.legend(loc="upper right")

plt.subplot(122)
plt.title("Predicted Clusters and Centroids")
sns.scatterplot(
    data=all_df,x="s_len",y="s_wid",
    hue="cluster",style = "cluster",
    markers="osPD",s=80
)
plt.legend(loc="upper right")

plt.show()

#### Part 6 — Hyperparameter tuning with multiple values of $k$

Fit the model for $k=2,3,4,5$. Record both metrics and determine whether they favour the same value of $k$.

In [ ]:
sil_coefs = []
distortions = []
K = np.arange(2, 6)

# TODO 9: For each k, fit your KMeans class, calculate both metrics,
# print the results, and append them to the lists above.
for k in K:
    pass

assert len(sil_coefs) == len(K) and len(distortions) == len(K), \
    "TODO 9: store one value of each metric for every k"

#### Part 7 - Using Elbow method to Find the optimal value of k

In [ ]:
# TODO 10: Plot distortion against k and label both axes.
# Mark each tested value of k on the x-axis.
pass

#### Part 8 -  Implementing K-means clustering with scikit learn

In [ ]:
from sklearn.cluster import KMeans as sklKMeans
from sklearn.metrics import silhouette_score as sk_silhouette_score

In [ ]:
# TODO 11: Fit scikit-learn's KMeans for k=3.
# For a reproducible comparison, set init="random", n_init=10, and random_state=0.
# Then calculate its silhouette score and distortion.
k = 3
sk_kmeans = None
ss = None
dist = None

# TODO 12: Print both metrics to five decimal places and compare them with
# your from-scratch implementation.

## b) Clustering using Gaussian Mixture Model(GMM)

### Program 1 — Implementing a GMM from scratch using the Expectation–Maximization algorithm

#### AIM

To implement a Gaussian Mixture Model (GMM) from scratch using the Expectation–Maximization (EM) algorithm and compare two methods for initializing the component means.

#### Formula:

##### Log-likelihood

For $K$ Gaussian components, the observed-data log-likelihood is

$$
\ell(\theta)=\sum_{i=1}^{N}\log\left[\sum_{k=1}^{K}\pi_k\,
\mathcal{N}(\mathbf{x}_i\mid\boldsymbol{\mu}_k,\boldsymbol{\Sigma}_k)\right].
$$

##### Auxiliary function

$$
Q(\theta,\theta^{(t-1)})=
\sum_i\sum_k r_{ik}\log\pi_k+
\sum_i\sum_k r_{ik}\log p(\mathbf{x}_i\mid\theta_k),
$$

where $r_{ik}=p(z_i=k\mid\mathbf{x}_i,\theta^{(t-1)})$ is the responsibility that component $k$ takes for observation $i$.

##### Responsibility (E-step)

$$
r_{ik}=\frac{\pi_k\,\mathcal{N}(\mathbf{x}_i\mid\boldsymbol{\mu}_k,\boldsymbol{\Sigma}_k)}
{\sum_{j=1}^{K}\pi_j\,\mathcal{N}(\mathbf{x}_i\mid\boldsymbol{\mu}_j,\boldsymbol{\Sigma}_j)}.
$$

For every observation $i$, the responsibilities must satisfy $\sum_k r_{ik}=1$.

##### Parameter updates (M-step)

Let $N_k=\sum_i r_{ik}$. Then

$$
\pi_k=\frac{N_k}{N},\qquad
\boldsymbol{\mu}_k=\frac{\sum_i r_{ik}\mathbf{x}_i}{N_k},\qquad
\boldsymbol{\Sigma}_k=
\frac{\sum_i r_{ik}(\mathbf{x}_i-\boldsymbol{\mu}_k)(\mathbf{x}_i-\boldsymbol{\mu}_k)^T}{N_k}.
$$

#### EM algorithm

1. Initialize the component weights, means, and covariance matrices.
2. **E-step:** calculate the posterior responsibility of every component for every observation.
3. **M-step:** update the weights, means, and covariance matrices using those responsibilities.
4. Evaluate the log-likelihood and repeat steps 2–3 until its improvement is no greater than the tolerance.

#### Part 1 - Defining the class for GMM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
from sklearn.cluster import KMeans as SklKMeans

In [ ]:
class GMM:
    def __init__(
        self, n_components: int, n_iters: int, tol: float,
        random_state: int = 0, init_params: str = "random",
        reg_covar: float = 1e-6,
    ):
        self.n_components = n_components
        self.n_iters = n_iters
        self.tol = tol
        self.init_params = init_params
        self.random_state = random_state
        self.reg_covar = reg_covar

    def fit(self, X, plot=False, plot_params=None):
        if plot_params is None:
            plot_params = {}

        m, n = X.shape
        self.X = X
        k = self.n_components

        self.resp = np.zeros((m, k))
        self.weights = np.full(k, 1 / k)

        # TODO 13: Initialize `self.means`.
        # - If init_params == "kmeans", fit SklKMeans and use its centers.
        # - If init_params == "random", select k distinct observations.
        # - Otherwise, raise ValueError for an unsupported method.
        self.means = None

        # TODO 14: Initialize every covariance matrix using the covariance of X.
        # Add reg_covar * identity for numerical stability.
        self.covs = None

        self.converged = False
        self.log_likelihood_trace = []

        if plot:
            fig, ax = plt.subplots(1, 4, figsize=(20, 5))
            fig.suptitle("GMM using Expectation–Maximization")
            self.draw(ax[0], "Initial components", **plot_params)

        for iteration in range(self.n_iters):
            # TODO 15: Perform one E-step and one M-step.
            pass

            # TODO 16: Append the current log-likelihood to its trace.
            pass

            if iteration == 0 and plot:
                self.draw(ax[1], "Components after 1 iteration", **plot_params)

            if iteration > 0:
                old_ll, new_ll = self.log_likelihood_trace[-2:]
                # TODO 17: Stop if the non-negative log-likelihood improvement
                # is no greater than `self.tol`.
                if False:
                    self.converged = True
                    break

        self.n_iter_ = iteration + 1

        if plot:
            self.draw(ax[2], f"Components after {self.n_iter_} iterations", **plot_params)
            ax[3].plot(self.log_likelihood_trace, "-o")
            ax[3].set_title("Log-likelihood")
            ax[3].set_xlabel("Iteration")
            plt.show()

        return self

    def _do_estep(self, X):
        # TODO 18: Calculate the unnormalized responsibilities
        # weights[k] * N(X | mean[k], cov[k]) for every component.
        pass

        # TODO 19: Normalize each row so that it sums to 1, and calculate
        # the total log-likelihood. Use a small lower bound to avoid log(0).
        pass

    def _do_mstep(self, X):
        # TODO 20: Calculate effective component counts N_k and update weights.
        resp_weights = None

        # TODO 21: Update the component means.
        pass

        # TODO 22: Update every covariance matrix using the weighted outer
        # products. Add reg_covar * identity to each matrix.
        pass

    def predict_proba(self, X):
        # TODO 23 (extension): Return responsibilities without changing the
        # fitted parameters or the stored training responsibilities.
        pass

    def predict(self, X):
        # TODO 24 (extension): Return the most probable component for each row.
        pass

    def draw(self, ax, title="", **plot_params):
        # Plot two-dimensional data, component means, and density contours.
        ax.set_title(title)
        ax.scatter(self.X[:, 0], self.X[:, 1], **plot_params)

        delta = 0.05
        x = np.arange(*ax.get_xlim(), delta)
        y = np.arange(*ax.get_ylim(), delta)
        x, y = np.meshgrid(x, y)
        colors = [f"C{(i + 1) % 10}" for i in range(self.n_components)]

        for i in range(self.n_components):
            mean, cov = self.means[i], self.covs[i]
            z = multivariate_normal(mean, cov).pdf(np.dstack([x, y]))
            ax.scatter(mean[0], mean[1], color=colors[i], marker="X", s=100)
            ax.contour(x, y, z, levels=[0.01], colors=colors[i])

#### Part 2 — Generating random data from a Gaussian mixture

The data-generation function is supplied. Read it carefully and identify where the means and positive-semidefinite covariance matrices are created.

In [ ]:
def gen_data(k=3, dim=2, points_per_cluster=200, lim=(-10, 10), seed=1):
    rng = np.random.default_rng(seed)
    samples = np.empty((k, points_per_cluster, dim))
    means = rng.uniform(lim[0], lim[1], size=(k, dim))

    for i in range(k):
        A = rng.random((dim, dim + 10))
        cov = A @ A.T
        samples[i] = rng.multivariate_normal(means[i], cov, points_per_cluster)

    X = samples.reshape(-1, dim)

    if dim == 2:
        plt.figure(figsize=(6, 5))
        plt.title("Generated Gaussian-mixture data")
        plt.scatter(X[:, 0], X[:, 1], s=3, alpha=0.4)
        plt.show()

    return X

In [ ]:
X = gen_data(k=3, dim=2, points_per_cluster=1000, seed=3)

#### Part 3 - Implementing GMM on generated random data

In [ ]:
# TODO 25: Create a 3-component GMM with random initialization, fit it to X,
# and display the progress plots. Use n_iters=100, tol=1e-4, random_state=5.
gmm_random = None

# After fitting, uncomment these diagnostic checks.
# assert np.allclose(gmm_random.weights.sum(), 1.0)
# assert np.allclose(gmm_random.resp.sum(axis=1), 1.0)
# assert np.all(np.diff(gmm_random.log_likelihood_trace) >= -1e-6)

#### Part 4 - Implementing GMM by using kmeans to initialize means

In [ ]:
# TODO 26: Repeat the fit with K-means initialization and random_state=1.
gmm_kmeans = None

# TODO 27: Compare the two runs using the final log-likelihood and number of
# iterations. Which initialization converged faster? Did both reach the same
# solution? Write your interpretation in the markdown cell below.

## Results and interpretation

Complete this section after running both algorithms.

1. **Best value of $k$ for Iris:** TODO — report the evidence from the elbow curve and silhouette scores.
2. **From-scratch vs scikit-learn K-means:** TODO — compare the two metric values and comment on initialization.
3. **Random vs K-means initialization for GMM:** TODO — compare convergence speed and final log-likelihood.
4. **K-means vs GMM:** TODO — explain one important difference between hard assignments and probabilistic responsibilities.
5. **Chemistry connection:** TODO — give one chemical dataset for which GMM clustering could be more informative than K-means (for example, overlapping conformer populations or spectra).